# SASHIMI-SI: matched CDM and SIDM subhalo structure

## Goal
Compare matched CDM and SIDM populations in a $10^{12}M_\odot$ host. Starting from
the same accretion nodes, plot the velocity-dependent interaction, weighted
$V_{max}$–$r_{max}$ distributions, collapsed mass function and one realization.
The physical parameters follow the historical `sample.ipynb`.

This is a physical example of the current ITAMAE-backed calculation. Each figure
is computed below from the selected model, with explicit units and population
cuts. A joint grid refinement at the end measures numerical sensitivity for this
example; it does not establish simulation calibration or an observational limit.

## Setup
From this repository's migration checkout, install the pinned development environment:
```sh
uv sync --extra demo
uv run --no-sync python -m ipykernel install --user --name sashimi-si-demo --display-name "sashimi-si demo"
```
Select that kernel and **Restart Kernel and Run All**. Allow several minutes for
the population and refinement calculations. No input files are downloaded by the
cells. The output directory is local to the notebook. Historical examples are
preserved in [archive/](archive/); the independent migration audit is in
[scientific_validation.ipynb](scientific_validation.ipynb).

In [ ]:
%matplotlib inline
import sys
import json
import time
import warnings
from pathlib import Path
from collections import Counter
from importlib.metadata import version
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from itamae.provenance import source_revision
from itamae.types import WeightedSubhaloCatalog
import itamae

plt.rcParams.update({"figure.figsize": (7.2, 4.5), "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.18,
                     "figure.constrained_layout.use": True})
COLORS = ["#0072B2", "#D55E00", "#009E73", "#555555"]
STYLES = ["-", "--", "-.", ":"]
output_dir = Path("outputs/usage_walkthrough")
output_dir.mkdir(parents=True, exist_ok=True)


def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n| " +
                     " | ".join(["---"] * len(headers)) + " |\n" +
                     "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))


def calculate(label, function):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter("always", RuntimeWarning)
        result = function()
    print(f"{label}: {time.perf_counter() - start:.1f} s")
    if observed:
        counts = Counter(f"{item.category.__name__}: {item.message}" for item in observed)
        table(["Recorded warning", "Occurrences"], counts.items())
    return result


def check_catalog(catalog):
    weights = catalog.weight_final
    assert np.all(np.isfinite(weights)) and np.all(weights >= 0)
    assert weights.sum() > 0
    mass = catalog.columns["m_bound"]
    assert np.all(np.isfinite(mass)) and np.all(mass >= 0)
    return {"nodes": len(catalog), "expected_survivors": float(weights.sum()),
            "bound_mass_fraction": float(catalog.weighted_sum(mass) / 1e12),
            "N_bound_gt_1e8": float(weights[mass > 1e8].sum())}


def mass_function(catalog, edges, selection=None, column="m_bound"):
    selected = catalog if selection is None else catalog.select(selection)
    counts, _ = selected.weighted_histogram(column, bins=edges)
    return np.sqrt(edges[:-1] * edges[1:]), counts / np.diff(np.log(edges))



def accretion_bin_edges(catalog, group=6):
    # Bin boundaries lie between accretion-grid nodes, avoiding bin/grid beating.
    log_nodes = np.log(np.unique(catalog.columns["m200_acc"]))
    spacing = np.diff(log_nodes)
    np.testing.assert_allclose(spacing, spacing[0], rtol=1e-10)
    interior = (log_nodes[:-1] + log_nodes[1:]) / 2
    return np.exp(np.r_[log_nodes[0] - spacing[0]/2, interior[group-1::group],
                        log_nodes[-1] + spacing[-1]/2])


def cumulative(values, weights, thresholds):
    return np.array([weights[values > threshold].sum() for threshold in thresholds])


def save_catalog(catalog, name):
    path = output_dir / (name + ".npz")
    catalog.to_npz(path)
    restored = WeightedSubhaloCatalog.from_npz(path)
    np.testing.assert_array_equal(restored.weight_final, catalog.weight_final)
    for column in catalog.columns:
        np.testing.assert_array_equal(restored.columns[column], catalog.columns[column])
    assert restored.metadata == catalog.metadata
    return path

import sashimi_si
components = [("sashimi-itamae", "itamae", itamae), ("sashimi-si", "sashimi-si", sashimi_si)]
provenance = {dist: {"version": version(dist), "source_revision": source_revision(name, module_file=mod.__file__)}
              for dist, name, mod in components}
table(["Component", "Version", "Source revision"],
      [(name, info["version"], info["source_revision"]) for name, info in provenance.items()])
print("Python", sys.version.split()[0], "| NumPy", version("numpy"), "| SciPy", version("scipy"))

## 1. Specify the self-interaction

Use $\sigma_0/m_\chi=147.1$ cm²/g and $w=24.33$ km/s. The displayed viscosity
cross section is a function of relative speed; these values are model inputs.

In [ ]:
from sashimi_si import SIDM_cross_section, SubhaloProperties
physical = dict(sigma0_m=147.1, w=24.33)
velocity = np.geomspace(1, 1000, 200)
interaction = SIDM_cross_section()
cross_section = interaction.sigma_viscosity(physical["sigma0_m"], physical["w"], velocity)
assert np.all(np.isfinite(cross_section)) and np.all(cross_section > 0)
fig, ax = plt.subplots()
ax.loglog(velocity, cross_section)
ax.axvline(physical["w"], color=COLORS[1], ls="--", label="w = 24.33 km/s")
ax.set(xlabel="Relative speed [km/s]", ylabel="Viscosity cross section [cm²/g]",
       title="Velocity-dependent SIDM interaction")
ax.legend()
plt.show()

## 2. Evolve matched populations

The common accretion domain is $M_{200,acc}=10^5$–$10^{11}M_\odot$ and
$0<z_{acc}\le5$. CDM and SIDM share accretion nodes, but have their own final
structure and survival weights. `weight_final` includes the selected survival
view. Native catalog lengths are Mpc and velocities are km/s.

In [ ]:
parameters = dict(M0=1e12, redshift=0., dz=.05, zmax=5., N_ma=192,
                  N_herm=10, N_hermNa=64, sigmalogc=.128, logmamin=5.,
                  logmamax=11., ct_th=0., Na_model=3, method="pert2_shanks")
model = SubhaloProperties(**physical)
catalogs = calculate("Matched population", lambda: model.subhalo_catalogs_calc(**parameters))
catalog, cdm = catalogs["sidm"], catalogs["cdm_reference"]
np.testing.assert_array_equal(catalog.columns["m200_acc"], cdm.columns["m200_acc"])
rows = []
for name, current in [("CDM", cdm), ("SIDM", catalog)]:
    metrics = check_catalog(current)
    rows.append((name, f"{metrics['expected_survivors']:.3f}",
                 f"{metrics['bound_mass_fraction']:.5f}", f"{metrics['N_bound_gt_1e8']:.3f}"))
table(["Model", "Expected survivors", "Bound mass / host mass", "N(bound mass > 1e8 Msun)"], rows)
print("Physics:", catalog.metadata["calculation_specification"])
mass_edges = np.geomspace(1e6, 1e11, 31)

## 3. Compare structure with correctly weighted distributions

Select **current bound mass** $>10^8M_\odot/h$, matching the historical example's
threshold. Each view uses its own survival weights. The common axes and color
scale show $d^2N/(d\log_{10}V_{max}\,d\log_{10}r_{max})$: expected count per dex².
Empty bins are masked rather than assigned a finite logarithmic density.

In [ ]:
from matplotlib.colors import LogNorm
threshold_mass = 1e8 / model.h
v_edges = np.geomspace(3, 160, 31)
r_edges = np.geomspace(.03, 60, 31)  # kpc
histograms = []
for view, kind in [(cdm, "cdm"), (catalog, "sidm")]:
    select = (view.columns["m_bound"] > threshold_mass) & (view.weight_final > 0)
    vx = view.columns[f"v_max_{kind}"][select]
    ry = 1000 * view.columns[f"r_max_{kind}"][select]
    weights = view.weight_final[select]
    counts, _, _ = np.histogram2d(vx, ry, bins=(v_edges, r_edges), weights=weights)
    area = np.diff(np.log10(v_edges))[:, None] * np.diff(np.log10(r_edges))[None, :]
    histograms.append(counts / area)
    print(f"{kind.upper()}: {weights.sum():.3f} selected; {counts.sum():.3f} inside displayed axes")
maximum = max(hist.max() for hist in histograms)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, hist, label in zip(axes, histograms, ["Matched CDM", "SIDM"]):
    mesh = ax.pcolormesh(v_edges, r_edges, np.ma.masked_less_equal(hist.T, 0),
                         norm=LogNorm(vmin=max(maximum*1e-4, 1e-4), vmax=maximum), cmap="viridis")
    ax.set(xscale="log", yscale="log", xlabel=r"$V_{max}$ [km/s]", title=label)
axes[0].set_ylabel(r"$r_{max}$ [kpc]")
fig.colorbar(mesh, ax=axes, label="Expected subhalos per dex²", shrink=.9)
fig.suptitle(r"Structure at $m_{bound}>10^8 M_\odot/h$")
plt.show()

## 4. Separate the collapsed population

The model diagnostic `collapse_time_ratio > 1` selects the collapsed SIDM view.
Its mass function uses SIDM weights, including SIDM survival. The denominator
for the reported collapsed fraction is the selected surviving population, not
an unweighted number of integration nodes.

In [ ]:
ratio = catalog.columns["collapse_time_ratio"]
collapsed = ratio > 1
fig, ax = plt.subplots()
for view, selection, label, color, style in [
    (cdm, None, "Matched CDM", COLORS[0], "--"),
    (catalog, None, "SIDM: all survivors", COLORS[1], "-"),
    (catalog, collapsed, "SIDM: collapse ratio > 1", COLORS[2], "-.")]:
    mass, dndlnm = mass_function(view, mass_edges, selection)
    ax.loglog(mass, np.where(dndlnm > 0, dndlnm, np.nan), style, color=color, label=label)
ax.set(xlabel=r"Bound mass [$M_\odot$]", ylabel=r"$dN/d\ln m$", title="Mass function and collapsed subset")
ax.legend()
plt.show()
resolved = catalog.columns["m_bound"] > threshold_mass
fraction = catalog.weight_final[resolved & collapsed].sum() / catalog.weight_final[resolved].sum()
print(f"Collapsed fraction above the displayed structural mass cut: {fraction:.3%}")

## 5. Generate one SIDM realization

Poisson sampling uses the catalog's expected-count intensity, with a fixed seed.
Color reports the collapse-time ratio; the displayed maximum is capped at one
only for color mapping, not in the stored catalog or the collapsed selection.

In [ ]:
realization = catalog.select(catalog.columns["m_bound"] > threshold_mass).poisson_realization(np.random.default_rng(20260911))
fig, ax = plt.subplots()
points = ax.scatter(realization["v_max_sidm"], 1000*realization["r_max_sidm"],
                    c=np.minimum(realization["collapse_time_ratio"], 1), vmin=0, vmax=1,
                    s=22, cmap="viridis")
ax.set(xscale="log", yscale="log", xlim=(v_edges[0], v_edges[-1]), ylim=(r_edges[0], r_edges[-1]),
       xlabel=r"$V_{max}$ [km/s]", ylabel=r"$r_{max}$ [kpc]", title="One SIDM realization")
fig.colorbar(points, ax=ax, label="Collapse-time ratio (color capped at 1)")
plt.show()
print("Drawn subhalos:", len(realization["m_bound"]))

## Checks: refine the same physical example

The refinement halves `dz` and increases mass and both Hermite grids. It retains
$\sigma_0/m_\chi$, $w$, the mass/redshift domain, survival rule and solver.
The table quantifies remaining numerical changes for the SIDM view. See
[resolution and state diagnostics](../docs/resolution-and-states.md) for model
validity boundaries and the separate solver comparison.

In [ ]:
refined_parameters = {**parameters, "dz": .025, "N_ma": 256, "N_herm": 14, "N_hermNa": 96}
refined_views = calculate("Refined matched population", lambda: SubhaloProperties(**physical).subhalo_catalogs_calc(**refined_parameters))
refined_catalog = refined_views["sidm"]

coarse_metrics, fine_metrics = check_catalog(catalog), check_catalog(refined_catalog)
refinement_rows = []
for metric in ("expected_survivors", "bound_mass_fraction", "N_bound_gt_1e8"):
    first, second = coarse_metrics[metric], fine_metrics[metric]
    delta = 100 * (second / first - 1)
    refinement_rows.append((metric, f"{first:.6g}", f"{second:.6g}", f"{delta:+.3f}%"))
table(["Quantity", "Displayed grid", "Refined grid", "Change"], refinement_rows)
fig, ax = plt.subplots()
for cat, label, style in [(catalog, "Displayed grid", "-"), (refined_catalog, "Refined grid", "--")]:
    mass, dndlnm = mass_function(cat, mass_edges)
    ax.loglog(mass, np.where(dndlnm > 0, dndlnm, np.nan), style, label=label)
ax.set(xlabel=r"Bound mass [$M_\odot$]", ylabel=r"$dN/d\ln m$", title="Sensitivity to joint numerical refinement")
ax.legend()
plt.show()
report = {"provenance": provenance, "parameters": parameters, "refined_parameters": refined_parameters,
          "displayed": coarse_metrics, "refined": fine_metrics}
(output_dir / "resolution-summary.json").write_text(json.dumps(report, indent=2) + "\n")
print("Catalog saved and checked:", save_catalog(catalog, "catalog"))
print("Refined catalog saved and checked:", save_catalog(refined_catalog, "catalog-refined"))

print("Matched CDM saved:", save_catalog(cdm, "cdm-reference"))

## Next steps

The weighted structural maps and mass functions replace the historical sample's
corresponding plots while preserving the current SIDM state and unit contracts.
The selected interaction is illustrative; this example does not infer a cross
section from observed dwarfs. Changes outside the calibrated evolution range
require the documented validity checks, not extrapolation of these figures.

Physical model: [Yang et al., *A Parametric Model for Self-Interacting Dark Matter
Halos*](https://arxiv.org/abs/2305.16176). The [scientific validation notebook](scientific_validation.ipynb)
compares the adopted implementation with independent references.